In [13]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/mt_dataset_pref1_b01.csv')

#df = df[df['participant_id']=='Yunt']
df = df.drop(columns=['participant_id'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14025 entries, 0 to 14024
Data columns (total 24 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   x1_linspace    14025 non-null  float64
 1   x1_brightness  14025 non-null  float64
 2   x1_symbol_m    14025 non-null  float64
 3   x1_symbol_c    14025 non-null  float64
 4   x1_symbol_s    14025 non-null  float64
 5   x1_symbol_y    14025 non-null  float64
 6   x1_symbol_f    14025 non-null  float64
 7   x1_symbol_j    14025 non-null  float64
 8   x1_location_0  14025 non-null  float64
 9   x1_location_1  14025 non-null  float64
 10  x1_location_2  14025 non-null  float64
 11  x1_location_3  14025 non-null  float64
 12  x2_linspace    14025 non-null  float64
 13  x2_brightness  14025 non-null  float64
 14  x2_symbol_m    14025 non-null  float64
 15  x2_symbol_c    14025 non-null  float64
 16  x2_symbol_s    14025 non-null  float64
 17  x2_symbol_y    14025 non-null  float64
 18  x2_sym

In [11]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -----------------------------
# 1) Колонки
# -----------------------------
x1_cols = [
    "x1_linspace", "x1_brightness",
    "x1_symbol_m", "x1_symbol_c", "x1_symbol_s", "x1_symbol_y", "x1_symbol_f", "x1_symbol_j",
    "x1_location_0", "x1_location_1", "x1_location_2", "x1_location_3",
]
x2_cols = [
    "x2_linspace", "x2_brightness",
    "x2_symbol_m", "x2_symbol_c", "x2_symbol_s", "x2_symbol_y", "x2_symbol_f", "x2_symbol_j",
    "x2_location_0", "x2_location_1", "x2_location_2", "x2_location_3",
]
assert len(x1_cols) == len(x2_cols)
F_DIM = len(x1_cols)


def prepare_arrays(df: pd.DataFrame):
    missing = [c for c in (x1_cols + x2_cols) if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in df: {missing}")
    X1 = df[x1_cols].to_numpy(dtype=np.float32)
    X2 = df[x2_cols].to_numpy(dtype=np.float32)
    return X1, X2


# -----------------------------
# 2) Симметризация: делаем метки
#    (x1,x2,y=1) и (x2,x1,y=0)
# -----------------------------
def symmetrize_with_labels(X1: np.ndarray, X2: np.ndarray):
    X_left  = np.vstack([X1, X2]).astype(np.float32)
    X_right = np.vstack([X2, X1]).astype(np.float32)
    y = np.concatenate([
        np.ones(len(X1), dtype=np.float32),
        np.zeros(len(X1), dtype=np.float32),
    ])
    return X_left, X_right, y


# -----------------------------
# 3) Dataset
# -----------------------------
class PairLabelDataset(Dataset):
    def __init__(self, XL: np.ndarray, XR: np.ndarray, y: np.ndarray):
        self.XL = torch.from_numpy(XL).float()
        self.XR = torch.from_numpy(XR).float()
        self.y  = torch.from_numpy(y).float()

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return self.XL[idx], self.XR[idx], self.y[idx]


# -----------------------------
# 4) Reward model r(x): MLP -> scalar
# -----------------------------
class RewardMLP(nn.Module):
    def __init__(self, in_dim: int, hidden=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)  # (B,)


# -----------------------------
# 5) Eval: loss + accuracy
# -----------------------------
@torch.no_grad()
def eval_loss_acc_auc(model, loader, criterion, device):
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    all_logits, all_y = [], []

    for xl, xr, yb in loader:
        xl, xr, yb = xl.to(device), xr.to(device), yb.to(device)
        logit = model(xl) - model(xr)
        loss = criterion(logit, yb)

        bs = xl.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

        probs = torch.sigmoid(logit)
        preds = (probs >= 0.5).float()
        correct += int((preds == yb).sum().item())

        all_logits.append(logit.detach().cpu().numpy())
        all_y.append(yb.detach().cpu().numpy())

    logits_np = np.concatenate(all_logits)
    y_np = np.concatenate(all_y)
    auc = roc_auc_score(y_np, logits_np)  # можно logits, можно probs
    return total_loss / total, correct / total, auc

@torch.no_grad()
def eval_loss_acc(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0

    for xl, xr, yb in loader:
        xl = xl.to(device)
        xr = xr.to(device)
        yb = yb.to(device)

        logit = model(xl) - model(xr)  # (B,)
        loss = criterion(logit, yb)

        bs = xl.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

        probs = torch.sigmoid(logit)
        preds = (probs >= 0.5).float()
        correct += int((preds == yb).sum().item())

    return total_loss / max(total, 1), correct / max(total, 1)


# -----------------------------
# 6) Train
# -----------------------------
def train_ranknet_with_labels(
    df: pd.DataFrame,
    *,
    test_size=0.2,
    random_state=0,
    batch_size=256,
    epochs=30,
    lr=3e-3,
    weight_decay=1e-4,
    hidden=(128, 64),
    dropout=0.1,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # (a) исходные массивы
    X1, X2 = prepare_arrays(df)

    # (b) split ДО симметризации (важно, чтобы не было утечки)
    X1_tr, X1_va, X2_tr, X2_va = train_test_split(
        X1, X2, test_size=test_size, random_state=random_state
    )

    # (c) scaler: обучаем на всех объектах train (и x1, и x2)
    scaler = StandardScaler()
    scaler.fit(np.vstack([X1_tr, X2_tr]))

    X1_tr = scaler.transform(X1_tr).astype(np.float32)
    X2_tr = scaler.transform(X2_tr).astype(np.float32)
    X1_va = scaler.transform(X1_va).astype(np.float32)
    X2_va = scaler.transform(X2_va).astype(np.float32)

    # (d) симметризация + метки
    XL_tr, XR_tr, y_tr = symmetrize_with_labels(X1_tr, X2_tr)
    XL_va, XR_va, y_va = symmetrize_with_labels(X1_va, X2_va)

    # (e) loaders
    train_loader = DataLoader(PairLabelDataset(XL_tr, XR_tr, y_tr),
                              batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(PairLabelDataset(XL_va, XR_va, y_va),
                            batch_size=batch_size, shuffle=False)

    # (f) model + loss + optim
    model = RewardMLP(in_dim=F_DIM, hidden=hidden, dropout=dropout).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # (g) loop
    for epoch in range(1, epochs + 1):
        model.train()
        for xl, xr, yb in train_loader:
            xl = xl.to(device)
            xr = xr.to(device)
            yb = yb.to(device)

            logit = model(xl) - model(xr)
            loss = criterion(logit, yb)

            optim.zero_grad(set_to_none=True)
            loss.backward()
            optim.step()

        tr_loss, tr_acc, tr_auc = eval_loss_acc_auc(model, train_loader, criterion, device)
        va_loss, va_acc, va_auc = eval_loss_acc_auc(model, val_loader, criterion, device)

        print(f"epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} auc {tr_auc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f} auc {va_auc:.4f}")

    return model, scaler, device


# -----------------------------
# 7) Инференс: P(x1 preferred x2)
# -----------------------------
@torch.no_grad()
def preference_prob(model, scaler, x1_vec, x2_vec, device=None):
    """
    x1_vec/x2_vec: shape (12,) в том же порядке, что x1_cols (но без префикса)
    """
    if device is None:
        device = next(model.parameters()).device

    x1 = np.asarray(x1_vec, dtype=np.float32).reshape(1, -1)
    x2 = np.asarray(x2_vec, dtype=np.float32).reshape(1, -1)

    x1 = scaler.transform(x1).astype(np.float32)
    x2 = scaler.transform(x2).astype(np.float32)

    x1t = torch.from_numpy(x1).to(device)
    x2t = torch.from_numpy(x2).to(device)

    logit = (model(x1t) - model(x2t)).item()
    prob = 1.0 / (1.0 + np.exp(-logit))
    return float(prob), float(logit)

In [12]:
model, scaler, device = train_ranknet_with_labels(df, epochs=100)

epoch 01 | train loss 0.4848 acc 0.7732 auc 0.8479 | val loss 0.5049 acc 0.7561 auc 0.8330
epoch 02 | train loss 0.4805 acc 0.7765 auc 0.8512 | val loss 0.4977 acc 0.7619 auc 0.8377
epoch 03 | train loss 0.4756 acc 0.7796 auc 0.8546 | val loss 0.4966 acc 0.7622 auc 0.8408
epoch 04 | train loss 0.4798 acc 0.7791 auc 0.8549 | val loss 0.5044 acc 0.7651 auc 0.8409
epoch 05 | train loss 0.4719 acc 0.7786 auc 0.8570 | val loss 0.4923 acc 0.7665 auc 0.8419
epoch 06 | train loss 0.4712 acc 0.7794 auc 0.8571 | val loss 0.4924 acc 0.7633 auc 0.8423
epoch 07 | train loss 0.4693 acc 0.7785 auc 0.8587 | val loss 0.4884 acc 0.7690 auc 0.8444
epoch 08 | train loss 0.4678 acc 0.7804 auc 0.8591 | val loss 0.4914 acc 0.7647 auc 0.8439
epoch 09 | train loss 0.4672 acc 0.7811 auc 0.8594 | val loss 0.4880 acc 0.7643 auc 0.8455
epoch 10 | train loss 0.4661 acc 0.7818 auc 0.8603 | val loss 0.4895 acc 0.7679 auc 0.8441
epoch 11 | train loss 0.4631 acc 0.7836 auc 0.8623 | val loss 0.4853 acc 0.7697 auc 0.8468

In [15]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# -----------------------------
# A) Колонки (по твоему df)
# -----------------------------
x1_cols = [
    "x1_linspace", "x1_brightness",
    "x1_symbol_m", "x1_symbol_c", "x1_symbol_s", "x1_symbol_y", "x1_symbol_f", "x1_symbol_j",
    "x1_location_0", "x1_location_1", "x1_location_2", "x1_location_3",
]
x2_cols = [
    "x2_linspace", "x2_brightness",
    "x2_symbol_m", "x2_symbol_c", "x2_symbol_s", "x2_symbol_y", "x2_symbol_f", "x2_symbol_j",
    "x2_location_0", "x2_location_1", "x2_location_2", "x2_location_3",
]

assert len(x1_cols) == len(x2_cols)
F_DIM = len(x1_cols)


# -----------------------------
# B) Dataset: возвращает (x1, x2)
# -----------------------------
class PairDataset(Dataset):
    def __init__(self, X1: np.ndarray, X2: np.ndarray):
        self.X1 = torch.from_numpy(X1).float()
        self.X2 = torch.from_numpy(X2).float()

    def __len__(self):
        return self.X1.shape[0]

    def __getitem__(self, idx):
        return self.X1[idx], self.X2[idx]


# -----------------------------
# C) Reward model r(x): MLP -> scalar
# -----------------------------
class RewardMLP(nn.Module):
    def __init__(self, in_dim: int, hidden=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, F)
        return self.net(x).squeeze(-1)  # (B,)


# -----------------------------
# D) RankNet loss (x1 always preferred)
#     L = -log(sigmoid(r1 - r2)) = softplus(-(r1-r2))
# -----------------------------
def ranknet_loss(r1: torch.Tensor, r2: torch.Tensor) -> torch.Tensor:
    return F.softplus(-(r1 - r2)).mean()


# -----------------------------
# E) Подготовка данных из df
# -----------------------------
def prepare_arrays(df: pd.DataFrame):
    missing = [c for c in (x1_cols + x2_cols) if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in df: {missing}")

    X1 = df[x1_cols].to_numpy(dtype=np.float32)
    X2 = df[x2_cols].to_numpy(dtype=np.float32)
    return X1, X2


# -----------------------------
# F) Eval: loss + метрики без меток
#    - acc@0: доля пар, где (r1-r2)>0
#    - acc@margin: более строгая (например margin=0.5)
#    - mean_diff: средняя разность (r1-r2)
# -----------------------------
@torch.no_grad()
def eval_metrics(model, loader, device, margin=0.0):
    model.eval()
    total_loss = 0.0
    total = 0

    correct = 0
    diff_sum = 0.0

    for x1b, x2b in loader:
        x1b = x1b.to(device)
        x2b = x2b.to(device)

        r1 = model(x1b)
        r2 = model(x2b)
        diff = (r1 - r2)

        loss = F.softplus(-diff).mean()

        bs = x1b.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

        correct += int((diff > margin).sum().item())
        diff_sum += float(diff.mean().item()) * bs

    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
        "mean_diff": diff_sum / max(total, 1),
    }


# -----------------------------
# G) Train
# -----------------------------
def train_ranknet(
    df: pd.DataFrame,
    *,
    test_size=0.2,
    random_state=42,
    batch_size=256,
    epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    hidden=(64, 32),
    dropout=0.2,
    margin_for_report=0.5,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) numpy arrays
    X1, X2 = prepare_arrays(df)

    # 2) split (ВАЖНО: по строкам)
    X1_tr, X1_va, X2_tr, X2_va = train_test_split(
        X1, X2, test_size=test_size, random_state=random_state
    )

    # 3) scaler: стандартизируем ФИЧИ одинаково для x1 и x2
    scaler = StandardScaler()
    scaler.fit(np.vstack([X1_tr, X2_tr]))
    X1_tr = scaler.transform(X1_tr).astype(np.float32)
    X2_tr = scaler.transform(X2_tr).astype(np.float32)
    X1_va = scaler.transform(X1_va).astype(np.float32)
    X2_va = scaler.transform(X2_va).astype(np.float32)

    # 4) loaders
    train_loader = DataLoader(PairDataset(X1_tr, X2_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(PairDataset(X1_va, X2_va), batch_size=batch_size, shuffle=False)

    # 5) model
    model = RewardMLP(in_dim=F_DIM, hidden=hidden, dropout=dropout).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # 6) loop
    for epoch in range(1, epochs + 1):
        model.train()
        for x1b, x2b in train_loader:
            x1b = x1b.to(device)
            x2b = x2b.to(device)

            r1 = model(x1b)
            r2 = model(x2b)
            loss = ranknet_loss(r1, r2)

            optim.zero_grad(set_to_none=True)
            loss.backward()
            optim.step()

        tr0 = eval_metrics(model, train_loader, device, margin=0.0)
        va0 = eval_metrics(model, val_loader, device, margin=0.0)
        vam = eval_metrics(model, val_loader, device, margin=margin_for_report)

        print(
            f"epoch {epoch:02d} | "
            f"train loss {tr0['loss']:.4f} acc@0 {tr0['acc']:.3f} meanΔ {tr0['mean_diff']:.3f} | "
            f"val loss {va0['loss']:.4f} acc@0 {va0['acc']:.3f} meanΔ {va0['mean_diff']:.3f} | "
            f"val acc@{margin_for_report} {vam['acc']:.3f}"
        )

    return model, scaler, device


# -----------------------------
# H) Инференс: P(x1 preferred x2)
# -----------------------------
@torch.no_grad()
def preference_prob(model, scaler, x1_vec, x2_vec, device=None):
    """
    x1_vec/x2_vec: shape (12,) в порядке колонок x1_cols/x2_cols (НО без префикса),
    т.е. как одна строка признаков для объекта.
    """
    if device is None:
        device = next(model.parameters()).device

    x1 = np.asarray(x1_vec, dtype=np.float32).reshape(1, -1)
    x2 = np.asarray(x2_vec, dtype=np.float32).reshape(1, -1)
    x1 = scaler.transform(x1).astype(np.float32)
    x2 = scaler.transform(x2).astype(np.float32)

    x1t = torch.from_numpy(x1).to(device)
    x2t = torch.from_numpy(x2).to(device)

    diff = (model(x1t) - model(x2t)).item()
    prob = 1.0 / (1.0 + np.exp(-diff))  # sigmoid(diff)
    return float(prob), float(diff)

In [16]:
model, scaler, device = train_ranknet(df, epochs=100, hidden=(64, 32), dropout=0.2)

epoch 01 | train loss 0.5104 acc@0 0.756 meanΔ 0.768 | val loss 0.5153 acc@0 0.754 meanΔ 0.758 | val acc@0.5 0.619
epoch 02 | train loss 0.5017 acc@0 0.761 meanΔ 0.869 | val loss 0.5055 acc@0 0.763 meanΔ 0.858 | val acc@0.5 0.642
epoch 03 | train loss 0.4981 acc@0 0.766 meanΔ 0.863 | val loss 0.5012 acc@0 0.765 meanΔ 0.852 | val acc@0.5 0.640
epoch 04 | train loss 0.4952 acc@0 0.765 meanΔ 0.899 | val loss 0.4988 acc@0 0.765 meanΔ 0.886 | val acc@0.5 0.642
epoch 05 | train loss 0.4929 acc@0 0.767 meanΔ 0.925 | val loss 0.4956 acc@0 0.768 meanΔ 0.913 | val acc@0.5 0.644
epoch 06 | train loss 0.4921 acc@0 0.768 meanΔ 0.862 | val loss 0.4952 acc@0 0.769 meanΔ 0.849 | val acc@0.5 0.636
epoch 07 | train loss 0.4896 acc@0 0.770 meanΔ 0.949 | val loss 0.4929 acc@0 0.768 meanΔ 0.933 | val acc@0.5 0.647
epoch 08 | train loss 0.4881 acc@0 0.773 meanΔ 0.959 | val loss 0.4907 acc@0 0.770 meanΔ 0.944 | val acc@0.5 0.653
epoch 09 | train loss 0.4882 acc@0 0.771 meanΔ 0.924 | val loss 0.4906 acc@0 0.7